# PHASE 1 — Analyse Exploratoire des Données (EDA)
## FRAUDX — Détection de fraude bancaire par IA au Togo

**Objectif :** Comprendre le dataset IEEE-CIS Fraud Detection, produire les figures pour le Chapitre III (section 3.3.1).

**Jours 3-7 du plan**

---
### Contenu :
- Jour 3 : Aperçu général, dimensions, types, mémoire
- Jour 4 : Distribution de isFraud (barplot + camembert)
- Jour 5 : Analyse des valeurs manquantes (missingno)
- Jour 6 : Analyses univariées/bivariées (TransactionAmt, cartes)
- Jour 7 : Variables temporelles + heatmap corrélations

In [ ]:
# === INSTALLATION DES DÉPENDANCES ===
!pip install pandas numpy matplotlib seaborn scikit-learn xgboost imbalanced-learn shap kagglehub optuna category_encoders missingno

In [ ]:
# === IMPORT DES BIBLIOTHÈQUES ===
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import missingno as msno
import warnings
warnings.filterwarnings('ignore')

sns.set_style('whitegrid')
plt.rcParams['figure.dpi'] = 120
plt.rcParams['figure.figsize'] = (12, 5)

print('Bibliothèques importées avec succès')

In [ ]:
# === MONTAGE GOOGLE DRIVE (Colab) ===
from google.colab import drive
drive.mount('/content/drive')
print('Drive monté avec succès')

---
## JOUR 3 — Aperçu général du dataset

**Objectif :** Charger les données, examiner les dimensions, les types, la mémoire.

In [ ]:
# === Téléchargement via kagglehub ===
import kagglehub

print('Téléchargement du dataset IEEE-CIS...')
path = kagglehub.dataset_download("ieee-fraud-detection")
print(f'Dataset téléchargé dans : {path}')

In [ ]:
# === Chargement des fichiers ===
import os

train_trans = pd.read_csv(os.path.join(path, 'train_transaction.csv'))
train_id = pd.read_csv(os.path.join(path, 'train_identity.csv'))

print(f'train_transaction : {train_trans.shape}')
print(f'train_identity : {train_id.shape}')

In [ ]:
# === Fusion sur TransactionID ===
df = train_trans.merge(train_id, on='TransactionID', how='left')
print(f'Fusionné : {df.shape}')
print(f'Colonnes : {df.columns.tolist()[:10]}... ({len(df.columns)} total)')

In [ ]:
# === Informations générales ===
print('=== df.info() ===')
df.info()

In [ ]:
# === df.describe() ===
df.describe()

In [ ]:
# === Mémoire utilisée ===
mem = df.memory_usage(deep=True).sum() / 1024**2
print(f'Mémoire totale : {mem:.2f} Mo')
print(f'Nombre de lignes : {len(df):,}')
print(f'Nombre de colonnes : {len(df.columns)}')

In [ ]:
# === Aperçu des 5 premières lignes ===
df.head()

In [ ]:
# === Types de colonnes ===
num_cols = df.select_dtypes(include=np.number).columns.tolist()
cat_cols = df.select_dtypes(include='object').columns.tolist()
print(f'Colonnes numériques : {len(num_cols)}')
print(f'Colonnes catégorielles (objets) : {len(cat_cols)}')

---
## JOUR 4 — Distribution de la variable cible (isFraud)

**Objectif :** Visualiser le déséquilibre des classes (~3,5% de fraude).

**Interprétation pour le mémoire :** Ce déséquilibre justifie l'usage de SMOTE et le choix de métriques adaptées (F1, AUC-PR, Recall) plutôt que l'Accuracy seule.

In [ ]:
# === Distribution de isFraud ===
fraud_counts = df['isFraud'].value_counts()
fraud_rate = df['isFraud'].mean() * 100

print('=== Distribution de isFraud ===')
print(fraud_counts)
print(f'\nTaux de fraude : {fraud_rate:.2f}%')
print(f'Taux de non-fraude : {100 - fraud_rate:.2f}%')

In [ ]:
# === Figure 3.x — Barplot + Camembert ===
fig, axes = plt.subplots(1, 2, figsize=(10, 4))

# Barplot
colors = ['steelblue', 'crimson']
bars = axes[0].bar(['Non fraude (0)', 'Fraude (1)'], fraud_counts.values, color=colors)
axes[0].set_ylabel('Nombre de transactions')
axes[0].set_title('Distribution des classes', fontweight='bold')
for bar, val in zip(bars, fraud_counts.values):
    axes[0].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 500,
                 f'{val:,}', ha='center', fontsize=10)

# Camembert
axes[1].pie(fraud_counts.values, labels=['Non fraude', 'Fraude'], autopct='%1.2f%%',
            colors=colors, startangle=90, explode=(0, 0.05))
axes[1].set_title('Proportion fraude vs non-fraude', fontweight='bold')

plt.suptitle('Figure 3.x — Déséquilibre des classes (isFraud)', y=1.02)
plt.tight_layout()
plt.show()

**Commentaire mémoire :** Le taux de fraude de 3,5% confirme le déséquilibre classique des datasets de détection de fraude. Ce ratio justifie :
- L'utilisation de SMOTE pour le rééquilibrage (Phase 2)
- Le choix du F1-Score et de l'AUC-PR comme métriques principales (Chapitre II)
- L'importance du Recall pour minimiser les faux négatifs (HS1)

---
## JOUR 5 — Analyse des valeurs manquantes

**Objectif :** Identifier les colonnes avec un taux de NaN élevé, visualiser avec missingno.

In [ ]:
# === Calcul du taux de NaN ===
missing = (df.isnull().sum() / len(df) * 100).sort_values(ascending=False)
missing_top = missing[missing > 0].head(30)
print('Top 30 colonnes avec le plus de valeurs manquantes :')
print(missing_top.to_string())

In [ ]:
# === Figure — Missingno matrix (échantillon) ===
msno.matrix(df.sample(5000, random_state=42))
plt.title('Figure 3.x — Matrice des valeurs manquantes (échantillon 5000 lignes)')
plt.show()

In [ ]:
# === Figure — Barplot des NaN par colonne ===
high_missing = missing[missing > 50]
fig, ax = plt.subplots(figsize=(10, max(4, len(high_missing)*0.3)))
ax.barh(range(len(high_missing)), high_missing.values, color='salmon')
ax.set_yticks(range(len(high_missing)))
ax.set_yticklabels(high_missing.index, fontsize=8)
ax.axvline(90, color='red', linestyle='--', label='Seuil 90% (suppression)')
ax.set_xlabel('Taux de valeurs manquantes (%)')
ax.set_title(f'Figure 3.x — Colonnes avec plus de 50% de NaN')
ax.legend()
plt.tight_layout()
plt.show()
print(f'\nColonnes à >90% NaN (supprimées en Phase 2) : {len(missing[missing > 90])}')

In [ ]:
# === Liste des colonnes à supprimer ===
cols_to_drop = missing[missing > 90].index.tolist()
print(f'Colonnes avec >90% de NaN ({len(cols_to_drop)} au total) :')
for c in cols_to_drop:
    print(f'  - {c} : {missing[c]:.1f}%')

**Commentaire mémoire :** Les colonnes avec plus de 90% de valeurs manquantes seront supprimées en Phase 2. Cette décision est justifiée par l'impossibilité d'imputer des variables quasi-vides sans introduire un bruit excessif. Les colonnes issues de `train_identity` (données d'appareil) sont particulièrement touchées, ce qui est attendu car toutes les transactions n'ont pas de données d'identité associées.

---
## JOUR 6 — Analyses univariées et bivariées

**Objectif :** Analyser TransactionAmt et les variables de carte (card1-card6, ProductCD) selon isFraud.

In [ ]:
# === Statistiques descriptives de TransactionAmt ===
print('=== Statistiques TransactionAmt ===')
print(df['TransactionAmt'].describe())
print(f'\nMédiane : {df["TransactionAmt"].median():.2f}')
print(f'Moyenne : {df["TransactionAmt"].mean():.2f}')

In [ ]:
# === Figure — Distribution du montant (log) ===
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Histogramme log
df_pos = df[df['isFraud'] == 1]['TransactionAmt']
df_neg = df[df['isFraud'] == 0]['TransactionAmt']

axes[0].hist(np.log1p(df_neg), bins=80, alpha=0.6, label='Non fraude', color='steelblue', density=True)
axes[0].hist(np.log1p(df_pos), bins=80, alpha=0.6, label='Fraude', color='crimson', density=True)
axes[0].set_xlabel('log(TransactionAmt + 1)')
axes[0].set_ylabel('Densité')
axes[0].set_title('Distribution du montant (échelle log)', fontweight='bold')
axes[0].legend()

# Boxplot
bp = axes[1].boxplot([df_neg, df_pos], labels=['Non fraude', 'Fraude'], widths=0.4,
                     patch_artist=True)
bp['boxes'][0].set_facecolor('steelblue')
bp['boxes'][1].set_facecolor('crimson')
axes[1].set_ylabel('TransactionAmt')
axes[1].set_title('Boxplot du montant par classe', fontweight='bold')
axes[1].set_yscale('log')

plt.suptitle('Figure 3.x — Analyse de TransactionAmt', y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
# === Analyse des cartes (card1-card6) ===
card_cols = [c for c in df.columns if c.startswith('card')]
print(f'Variables de carte disponibles : {card_cols}')

# ProductCD
if 'ProductCD' in df.columns:
    print('\n=== ProductCD vs isFraud ===')
    ct = pd.crosstab(df['ProductCD'], df['isFraud'], normalize='index') * 100
    ct.columns = ['Non fraude (%)', 'Fraude (%)']
    print(ct.round(2))

In [ ]:
# === Figure — Taux de fraude par ProductCD ===
if 'ProductCD' in df.columns:
    fraud_by_product = df.groupby('ProductCD')['isFraud'].mean().sort_values() * 100
    fig, ax = plt.subplots(figsize=(8, 4))
    bars = ax.bar(fraud_by_product.index, fraud_by_product.values, color='coral', edgecolor='black')
    ax.set_xlabel('ProductCD')
    ax.set_ylabel('Taux de fraude (%)')
    ax.set_title('Figure 3.x — Taux de fraude par type de produit (ProductCD)', fontweight='bold')
    for bar, val in zip(bars, fraud_by_product.values):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.1, f'{val:.2f}%',
                ha='center', fontsize=9)
    plt.tight_layout()
    plt.show()

In [ ]:
# === Analyse card4 (type de carte) ===
if 'card4' in df.columns:
    print('=== card4 (type de carte) vs isFraud ===')
    ct_card4 = pd.crosstab(df['card4'], df['isFraud'], normalize='index') * 100
    ct_card4.columns = ['Non fraude (%)', 'Fraude (%)']
    print(ct_card4.round(2))

In [ ]:
# === Figure — Taux de fraude par type de carte ===
if 'card4' in df.columns:
    fraud_by_card4 = df.groupby('card4')['isFraud'].mean().sort_values() * 100
    fig, ax = plt.subplots(figsize=(8, 4))
    bars = ax.bar(fraud_by_card4.index, fraud_by_card4.values, color='mediumseagreen', edgecolor='black')
    ax.set_xlabel('card4 (Type de carte)')
    ax.set_ylabel('Taux de fraude (%)')
    ax.set_title('Figure 3.x — Taux de fraude par type de carte', fontweight='bold')
    for bar, val in zip(bars, fraud_by_card4.values):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.1, f'{val:.2f}%',
                ha='center', fontsize=9)
    plt.tight_layout()
    plt.show()

In [ ]:
# === Analyse card6 (bancaire/crédit) ===
if 'card6' in df.columns:
    print('=== card6 (débit/crédit) vs isFraud ===')
    ct_card6 = pd.crosstab(df['card6'], df['isFraud'], normalize='index') * 100
    ct_card6.columns = ['Non fraude (%)', 'Fraude (%)']
    print(ct_card6.round(2))

**Commentaire mémoire — Correspondance mobile money Togo :**

| Variable IEEE-CIS | Équivalent conceptuel Togo |
|---|---|
| `TransactionAmt` | Montant recharge/transfert/paiement mobile money (FCFA) |
| `card1`-`card6` | Identifiant SIM / device (IMEI) |
| `ProductCD` | Type d'opération (recharge, transfert, paiement marchand, retrait agent) |
| `C1`-`C14` | Fréquence des transactions par agent/numéro |
| `D1`-`D15` | Délai depuis dernière transaction (indicateur SIM swap) |

Cette correspondance est discutée dans la section 3.3.3 du mémoire.

---
## JOUR 7 — Variables temporelles + Heatmap corrélations

**Objectif :** Extraire l'heure/jour depuis TransactionDT, visualiser les distributions temporelles de la fraude, et produire la heatmap des corrélations.

In [ ]:
# === Conversion de TransactionDT en temps réel ===
# TransactionDT est en secondes depuis le 01/12/2017 00:00:00
import datetime

start_date = datetime.datetime(2017, 12, 1)
df['TransactionDT_dt'] = df['TransactionDT'].apply(lambda x: start_date + datetime.timedelta(seconds=x))
df['hour'] = df['TransactionDT_dt'].dt.hour
df['dayofweek'] = df['TransactionDT_dt'].dt.dayofweek
df['day'] = df['TransactionDT_dt'].dt.day

print('Colonnes temporelles créées : hour, dayofweek, day')
print(df[['TransactionDT', 'TransactionDT_dt', 'hour', 'dayofweek']].head())

In [ ]:
# === Figure — Fraude par heure ===
fig, axes = plt.subplots(1, 2, figsize=(14, 4))

# Volume par heure
hourly_fraud = df.groupby('hour')['isFraud'].mean() * 100
hourly_count = df.groupby('hour')['isFraud'].count()

axes[0].plot(hourly_count.index, hourly_count.values, marker='o', color='steelblue', linewidth=1.5)
axes[0].set_xlabel('Heure')
axes[0].set_ylabel('Volume de transactions')
axes[0].set_title('Volume de transactions par heure', fontweight='bold')
axes[0].set_xticks(range(24))

# Taux de fraude par heure
axes[1].plot(hourly_fraud.index, hourly_fraud.values, marker='o', color='crimson', linewidth=1.5)
axes[1].set_xlabel('Heure')
axes[1].set_ylabel('Taux de fraude (%)')
axes[1].set_title('Taux de fraude par heure', fontweight='bold')
axes[1].set_xticks(range(24))

plt.suptitle('Figure 3.x — Analyse temporelle des transactions', y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
# === Figure — Fraude par jour de la semaine ===
dow_labels = ['Lun', 'Mar', 'Mer', 'Jeu', 'Ven', 'Sam', 'Dim']
dow_fraud = df.groupby('dayofweek')['isFraud'].mean() * 100

fig, ax = plt.subplots(figsize=(8, 4))
bars = ax.bar(dow_labels, dow_fraud.values, color='mediumpurple', edgecolor='black')
ax.set_xlabel('Jour de la semaine')
ax.set_ylabel('Taux de fraude (%)')
ax.set_title('Figure 3.x — Taux de fraude par jour de la semaine', fontweight='bold')
for bar, val in zip(bars, dow_fraud.values):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.02, f'{val:.2f}%',
            ha='center', fontsize=9)
plt.tight_layout()
plt.show()

In [ ]:
# === Heatmap des corrélations (top 30 variables numériques) ===
# Sélectionner les colonnes numériques les plus corrélées à isFraud
numeric_df = df.select_dtypes(include=[np.number])
corr_with_fraud = numeric_df.corr()['isFraud'].abs().sort_values(ascending=False)
top30 = corr_with_fraud.index[:30]

# Matrice de corrélation
corr_matrix = numeric_df[top30].corr()

fig, ax = plt.subplots(figsize=(14, 12))
mask = np.triu(np.ones_like(corr_matrix, dtype=bool))
sns.heatmap(corr_matrix, mask=mask, cmap='RdBu_r', center=0, annot=False,
            square=True, linewidths=0.5, cbar_kws={'shrink': 0.8})
ax.set_title('Figure 3.x — Heatmap des corrélations (top 30 variables)', fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# === Top 10 corrélations à isFraud ===
top10_corr = corr_with_fraud[1:11]  # skip isFraud lui-même
print('Top 10 variables les plus corrélées à isFraud :')
for var, corr_val in top10_corr.items():
    direction = 'positive' if numeric_df[var].corr(df['isFraud']) > 0 else 'négative'
    print(f'  {var}: {corr_val:.4f} ({direction})')

---
## Synthèse de la Phase 1 — Éléments pour le mémoire

### Figures produites :
1. **Figure 3.x** — Distribution des classes (isFraud) : barplot + camembert → Jour 4
2. **Figure 3.x** — Matrice des valeurs manquantes (missingno) → Jour 5
3. **Figure 3.x** — Colonnes avec plus de 50% de NaN → Jour 5
4. **Figure 3.x** — Analyse du montant (histogramme log + boxplot) → Jour 6
5. **Figure 3.x** — Taux de fraude par ProductCD → Jour 6
6. **Figure 3.x** — Taux de fraude par type de carte (card4) → Jour 6
7. **Figure 3.x** — Analyse temporelle (volume + fraude par heure) → Jour 7
8. **Figure 3.x** — Taux de fraude par jour de la semaine → Jour 7
9. **Figure 3.x** — Heatmap des corrélations (top 30 variables) → Jour 7

### Chiffres clés :
- Taux de fraude : **3,5%**
- Dimensions : **~590 000 transactions**, **~400 variables**
- Colonnes >90% NaN : **variable** (à quantifier avec les résultats ci-dessus)
- Variable la plus corrélée à isFraud : **variable** (à relever du top10)